# 03 -- Genotype-alone model
Trains and compares **five** genotype-alone variants on the per-hybrid
marginal mean-yield target: **GBLUP** (VanRaden-kinship kernel ridge, the
baseline), **MLP-1** (one hidden layer, group-lasso sparsity on the input
layer -- "lasso" variant), **MLP-2** (two hidden layers, group-lasso
sparsity on both), **MLP-2-L2** (same two-hidden-layer architecture as
MLP-2, but standard L2 regularization on both layers instead of group
lasso, fixed lambda=0.00001, not swept), and **MLP-2-no-reg** (same
architecture again, no regularization at all) -- so the L2 and no-reg
variants isolate the effect of the regularization TYPE, holding
architecture fixed against mlp2_sparse.

All model/training/evaluation logic lives in `scripts/genotype_models.py`,
`scripts/training.py`, `scripts/evaluation.py` -- this notebook is execution
and comparison only, per project convention (see `scripts/README.md`).

**MLP input:** standardized (per-marker, train-fold-only z-score) dosage
values. This was reverted from an earlier version that used raw {0, 0.5, 1}
input directly -- the raw-input run showed a marked accuracy drop across
every MLP variant (even `mlp2_no_reg`, no regularization, underfit its own
training data), which points at an optimization problem rather than a
capacity one: real SNP markers are heavily allele-frequency-skewed (many
near-monomorphic), and un-centered, unequally-scaled raw input is a classic
cause of slow/poor neural-net convergence. This run tests that theory
directly by reverting to standardized input with everything else held
fixed. GBLUP is unaffected either way -- its kinship computation does its
own allele-frequency centering internally, independent of this choice.

**Sparsity mechanism (worth understanding before reading results, applies
to mlp1_lasso/mlp2_sparse only):** with a hidden layer, each input marker
feeds many hidden units, so plain elementwise L1 has no way to zero out a
whole marker's column of weights together -- it just shrinks entries a
little each, and "sparsity" never actually shows up even as accuracy
degrades from the shrinkage (confirmed empirically before landing on this
design). The fix is **group lasso**: penalize each marker's entire weight
column by its L2 norm, applied via an explicit **proximal (soft-
thresholding) step once per epoch** rather than as a gradient term --
Adam's adaptive per-parameter scaling turned out to fight a gradient-based
penalty badly (non-monotonic, unstable sparsity-vs-lambda), so training
uses **SGD + momentum + gradient clipping** instead. mlp2_l2 and
mlp2_no_reg don't have this problem -- L2 is a smooth penalty (and "no
penalty" is trivially smooth), so both train with plain **Adam**, no
proximal step needed. See `fit_mlp`/`fit_mlp_adam` and
`apply_group_lasso_proximal`'s docstrings in `scripts/` for the full
reasoning.

**Target and reliability weighting** (confirmed in `SESSION_MEMORY.md`):
per-hybrid marginal mean yield, sample-weighted by (capped) n_envs_tested so
the heavily-replicated check hybrids (max 259 environments) don't dominate
the loss relative to hybrids tested once. Applied identically across all
five variants via the same `reliability_weights` function, so the model
comparison isn't confounded by different weighting treatment.

**CV scheme:** leave-one-year-out, but only **5 evenly-spaced years**
(2014, 2016, 2018, 2020, 2022) rather than all 10, to cut compute given the
larger variant count -- adjust `CV_YEARS` below if a different subset is
preferred. For each held-out year, the per-hybrid target is recomputed
using ONLY the other years' data (train fold), and evaluated against that
hybrid's actual mean yield IN the held-out year (the year-specific truth).
Final models are refit on all of 2014-2023 and evaluated against the true
2024 test set.

## Setup
Colab detection, Drive mounting, `BASE_PATH` resolution, and `sys.path`
so `scripts/` imports directly as a namespace package.

In [1]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/g2f_effect_decomposition')
else:
    BASE_PATH = Path(os.environ.get('G2F_BASE_PATH', '.')).resolve()

sys.path.insert(0, str(BASE_PATH))

DATA_RAW = BASE_PATH / 'data' / 'raw'
DATA_PROCESSED = BASE_PATH / 'data' / 'processed'
RESULTS_DIR = BASE_PATH / 'results' / 'genotype_model'
MODELS_DIR = RESULTS_DIR / 'checkpoints'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"IN_COLAB: {IN_COLAB}")
print(f"BASE_PATH: {BASE_PATH}")

Mounted at /content/drive
IN_COLAB: True
BASE_PATH: /content/drive/MyDrive/g2f_effect_decomposition


In [2]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device (training only):', DEVICE)

Device (training only): cuda


In [3]:
import numpy as np
import pandas as pd

from scripts.genotype_models import (
    vanraden_allele_freq, vanraden_denominator, vanraden_kernel,
    build_variant, l2_penalty,
)
from scripts.training import reliability_weights, make_loader, fit_mlp, fit_mlp_adam, fit_gblup
from scripts.evaluation import evaluate_predictions, effective_markers

pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

HIDDEN_DIMS_MLP1 = [128]
HIDDEN_DIMS_MLP2 = [256, 64]
L1_GRID = [0.001, 0.005, 0.01, 0.02, 0.03, 0.05]  # calibrated for the proximal (once-per-epoch) group-lasso step
MLP2_L2_LAMBDA = 0.00001  # fixed, not swept -- mlp2_l2 uses this single value
RIDGE_ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]

BATCH_SIZE = 256
NUM_EPOCHS = 200
PATIENCE = 20
LR = 0.01  # SGD+momentum, for the group-lasso variants (mlp1_lasso, mlp2_sparse) -- see fit_mlp docstring for why
ADAM_LR = 1e-3  # Adam, for the smooth-penalty variants (mlp2_l2, mlp2_no_reg) -- see fit_mlp_adam docstring

# MLP input: raw {0, 0.5, 1} dosage values, NOT standardized -- applies to
# all four MLP variants below. GBLUP is unaffected (its kinship computation
# does its own allele-frequency centering internally, unrelated to this).

## Load data
Reloads the raw trait and genotype files directly (same pattern as
`01_effect_representations.ipynb`) rather than depending on that notebook's
saved, target-filtered dosage matrix -- the CV loop here needs the full
dosage matrix (train + test hybrids) and the per-plot trait table to
recompute per-fold targets, not just the precomputed exploratory target.

In [4]:
TRAIN_DIR = DATA_RAW / 'Training_data'
TEST_DIR = DATA_RAW / 'Testing_data'

trait_df = pd.read_csv(TRAIN_DIR / '1_Training_Trait_Data_2014_2023.csv')
geno_num = pd.read_csv(TRAIN_DIR / '5_Genotype_Data_All_2014_2025_Hybrids_numerical.txt',
                        sep='\t', skiprows=1)
test_observed_df = pd.read_csv(TEST_DIR / '7_Testing_Observed_Values.csv')

geno_id_col = geno_num.columns[0]
geno_num = geno_num.set_index(geno_id_col)
geno_num.index = geno_num.index.astype(str)

# Missing genotype calls must be imputed before any kinship/MLP use -- median
# per marker, consistent with the imputation approach used elsewhere in the
# project. Fit per-fold below for the CV loop; this global version is only
# used to establish which hybrids/markers are usable at all.
print(f"trait_df: {trait_df.shape}")
print(f"geno_num: {geno_num.shape} (missing calls: {geno_num.isna().sum().sum()})")
print(f"test_observed_df: {test_observed_df.shape}")

trait_df: (173960, 26)
geno_num: (5899, 2425) (missing calls: 455228)
test_observed_df: (9486, 3)


In [5]:
trait_clean = trait_df.dropna(subset=['Yield_Mg_ha']).copy()
trait_clean['Hybrid'] = trait_clean['Hybrid'].astype(str)

genotyped_hybrids = set(geno_num.index)
trait_genotyped = trait_clean[trait_clean['Hybrid'].isin(genotyped_hybrids)].copy()

print(f"trait_genotyped: {len(trait_genotyped)} rows, "
      f"{trait_genotyped['Hybrid'].nunique()} unique genotyped hybrids, "
      f"years {sorted(trait_genotyped['Year'].unique())}")

trait_genotyped: 163026 rows, 4938 unique genotyped hybrids, years [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


In [6]:
def build_hybrid_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Per-hybrid marginal mean yield and environment-count reliability,
    matching 01_effect_representations.ipynb -- reimplemented here (not
    imported) since it's a two-line groupby with no other shared state.
    """
    grouped = df.groupby('Hybrid')['Yield_Mg_ha']
    n_envs = df.groupby('Hybrid')['Env'].nunique()
    return pd.DataFrame({
        'hybrid_mean_yield': grouped.mean(),
        'n_envs_tested': n_envs,
    })

## Leave-one-year-out CV
For each held-out year: train target computed from the other years only,
evaluated against that hybrid's actual mean yield in the held-out year.
Genotype dosage is median-imputed and the kinship allele frequency is
estimated using the training fold only, in both cases -- fold-safe.

In [7]:
def prepare_fold(held_out_year: int, trait_genotyped: pd.DataFrame,
                  geno_num: pd.DataFrame) -> dict:
    """Builds train/val (X, y, weights) for one leave-one-year-out fold.

    X is median-imputed AND standardized (per-marker, train-fold-only
    mean/std) for MLP input -- reverted back from raw {0, 0.5, 1} to test
    whether standardization explains the accuracy drop observed with raw
    input. GBLUP is unaffected either way -- it uses X_train_raw/X_val_raw
    directly, since its kinship computation does its own allele-frequency
    centering internally. Validation hybrids need not have appeared in
    training -- genotype-based prediction should generalize to any hybrid
    with marker coverage.
    """
    train_rows = trait_genotyped[trait_genotyped['Year'] != held_out_year]
    val_rows = trait_genotyped[trait_genotyped['Year'] == held_out_year]

    train_targets = build_hybrid_targets(train_rows)
    val_targets = val_rows.groupby('Hybrid')['Yield_Mg_ha'].mean()

    train_hybrids = train_targets.index.tolist()
    val_hybrids = [h for h in val_targets.index if h in geno_num.index]

    marker_median = geno_num.loc[train_hybrids].median()
    geno_imputed = geno_num.fillna(marker_median)

    X_train_raw = geno_imputed.loc[train_hybrids].to_numpy(dtype=np.float64)
    X_val_raw = geno_imputed.loc[val_hybrids].to_numpy(dtype=np.float64)

    marker_mean = X_train_raw.mean(axis=0)
    marker_std = X_train_raw.std(axis=0)
    marker_std[marker_std == 0] = 1.0

    return {
        'X_train_raw': X_train_raw, 'X_val_raw': X_val_raw,
        'X_train_std': (X_train_raw - marker_mean) / marker_std,
        'X_val_std': (X_val_raw - marker_mean) / marker_std,
        'y_train': train_targets.loc[train_hybrids, 'hybrid_mean_yield'].to_numpy(dtype=np.float32),
        'y_val': val_targets.loc[val_hybrids].to_numpy(dtype=np.float32),
        'n_envs_train': train_targets.loc[train_hybrids, 'n_envs_tested'].to_numpy(),
        'train_hybrids': train_hybrids, 'val_hybrids': val_hybrids,
    }


years = sorted(trait_genotyped['Year'].unique())
CV_YEARS = years[::2]  # 5 evenly-spaced years instead of all 10, per current request
print(f"All years: {years}")
print(f"CV_YEARS (evenly spaced, 5 folds): {CV_YEARS}")

All years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
CV_YEARS (evenly spaced, 5 folds): [np.int64(2014), np.int64(2016), np.int64(2018), np.int64(2020), np.int64(2022)]


In [8]:
def run_gblup_fold(fold: dict) -> dict:
    allele_freq = vanraden_allele_freq(fold['X_train_raw'])
    denom = vanraden_denominator(allele_freq)
    G_train = vanraden_kernel(fold['X_train_raw'], fold['X_train_raw'], allele_freq, denom)
    G_val = vanraden_kernel(fold['X_val_raw'], fold['X_train_raw'], allele_freq, denom)

    weights = reliability_weights(fold['n_envs_train'])
    model, best_alpha, _ = fit_gblup(G_train, fold['y_train'], weights,
                                      G_val, fold['y_val'], RIDGE_ALPHA_GRID)

    pred_train = model.predict(G_train)
    pred_val = model.predict(G_val)
    return {
        'variant': 'gblup', 'best_hparam': best_alpha,
        'train_metrics': evaluate_predictions(fold['y_train'], pred_train),
        'val_metrics': evaluate_predictions(fold['y_val'], pred_val),
        'n_effective_markers': fold['X_train_raw'].shape[1],  # GBLUP uses all markers
    }


def run_mlp_fold(fold: dict, variant_name: str, hidden_dims_mlp1: list[int],
                  hidden_dims_mlp2: list[int]) -> list[dict]:
    """Trains one group-lasso variant across the full L1_GRID and returns
    EVERY lambda's result, rather than picking a single 'best' one here --
    selecting by validation accuracy alone always favors the weakest lambda
    (some accuracy cost is inherent to sparsity), which would silently
    defeat the point of asking for an L1-sparse model in the first place.
    Selection happens later, via the one-SE rule in `select_lambda_one_se_rule`.
    """
    weights_train = reliability_weights(fold['n_envs_train'])
    weights_val = np.ones(len(fold['y_val']), dtype=np.float32)  # val loss unweighted -- true generalization signal

    results = []
    for lam in L1_GRID:
        model, l1_idx = build_variant(variant_name, fold['X_train_std'].shape[1],
                                       hidden_dims_mlp1, hidden_dims_mlp2)
        train_loader = make_loader(fold['X_train_std'], fold['y_train'], weights_train,
                                    BATCH_SIZE, shuffle=True)
        val_loader = make_loader(fold['X_val_std'], fold['y_val'], weights_val,
                                  BATCH_SIZE, shuffle=False)
        model, _ = fit_mlp(model, train_loader, val_loader, l1_idx, lam, LR,
                            NUM_EPOCHS, PATIENCE, DEVICE, verbose=False)

        model.eval()
        with torch.no_grad():
            pred_train = model(torch.as_tensor(fold['X_train_std'], dtype=torch.float32).to(DEVICE)).cpu().numpy()
            pred_val = model(torch.as_tensor(fold['X_val_std'], dtype=torch.float32).to(DEVICE)).cpu().numpy()

        results.append({
            'variant': variant_name, 'l1_lambda': lam,
            'train_metrics': evaluate_predictions(fold['y_train'], pred_train),
            'val_metrics': evaluate_predictions(fold['y_val'], pred_val),
            'n_effective_markers': effective_markers(model, layer_idx=0),
        })
    return results


def run_mlp_smooth_fold(fold: dict, variant_name: str) -> dict:
    """Trains one non-group-lasso MLP variant -- 'mlp2_l2' (fixed lambda,
    L2 penalty on both hidden layers) or 'mlp2_no_reg' (no penalty at all)
    -- with Adam. A single run, no lambda grid, since both use one fixed
    configuration rather than a sparsity sweep. Both share MLP-2's
    architecture (two hidden layers) so the only difference from
    mlp2_sparse is the regularization type/strength.
    """
    weights_train = reliability_weights(fold['n_envs_train'])
    weights_val = np.ones(len(fold['y_val']), dtype=np.float32)

    model, layer_idx = build_variant(variant_name, fold['X_train_std'].shape[1],
                                      HIDDEN_DIMS_MLP1, HIDDEN_DIMS_MLP2)
    train_loader = make_loader(fold['X_train_std'], fold['y_train'], weights_train, BATCH_SIZE, shuffle=True)
    val_loader = make_loader(fold['X_val_std'], fold['y_val'], weights_val, BATCH_SIZE, shuffle=False)

    penalty_fn = (lambda m: l2_penalty(m, layer_idx, MLP2_L2_LAMBDA)) if variant_name == 'mlp2_l2' else None
    model, _ = fit_mlp_adam(model, train_loader, val_loader, penalty_fn, ADAM_LR,
                             NUM_EPOCHS, PATIENCE, DEVICE, verbose=False)

    model.eval()
    with torch.no_grad():
        pred_train = model(torch.as_tensor(fold['X_train_std'], dtype=torch.float32).to(DEVICE)).cpu().numpy()
        pred_val = model(torch.as_tensor(fold['X_val_std'], dtype=torch.float32).to(DEVICE)).cpu().numpy()

    return {
        'variant': variant_name,
        'l1_lambda': MLP2_L2_LAMBDA if variant_name == 'mlp2_l2' else None,
        'train_metrics': evaluate_predictions(fold['y_train'], pred_train),
        'val_metrics': evaluate_predictions(fold['y_val'], pred_val),
        'n_effective_markers': effective_markers(model, layer_idx=0),
    }

In [9]:
fold_results = []
for held_out_year in CV_YEARS:
    print(f"=== Held-out year {held_out_year} ===")
    fold = prepare_fold(held_out_year, trait_genotyped, geno_num)
    print(f"  train hybrids: {len(fold['train_hybrids'])}, val hybrids: {len(fold['val_hybrids'])}")

    gblup_result = run_gblup_fold(fold)
    gblup_result['held_out_year'] = held_out_year
    fold_results.append(gblup_result)
    print(f"  gblup        val_r={gblup_result['val_metrics']['pearson_r']:.3f} "
          f"val_rmse={gblup_result['val_metrics']['rmse']:.3f}")

    for variant_name in ['mlp1_lasso', 'mlp2_sparse']:
        variant_results = run_mlp_fold(fold, variant_name, HIDDEN_DIMS_MLP1, HIDDEN_DIMS_MLP2)
        for r in variant_results:
            r['held_out_year'] = held_out_year
        fold_results.extend(variant_results)
        best_by_r = max(variant_results, key=lambda r: r['val_metrics']['pearson_r'])
        print(f"  {variant_name:12s} best val_r={best_by_r['val_metrics']['pearson_r']:.3f} "
              f"(lambda={best_by_r['l1_lambda']:.0e}) -- full grid of {len(variant_results)} "
              f"lambdas recorded, see comparison table below")

    for variant_name in ['mlp2_l2', 'mlp2_no_reg']:
        smooth_result = run_mlp_smooth_fold(fold, variant_name)
        smooth_result['held_out_year'] = held_out_year
        fold_results.append(smooth_result)
        print(f"  {variant_name:12s} val_r={smooth_result['val_metrics']['pearson_r']:.3f} "
              f"val_rmse={smooth_result['val_metrics']['rmse']:.3f}")

=== Held-out year 2014 ===
  train hybrids: 4414, val hybrids: 850
  gblup        val_r=0.325 val_rmse=1.370


/content/drive/MyDrive/g2f_effect_decomposition/scripts/evaluation.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(pearsonr(y_true, y_pred)[0])


  mlp1_lasso   best val_r=0.373 (lambda=1e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_sparse  best val_r=0.293 (lambda=1e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_l2      val_r=0.365 val_rmse=1.522
  mlp2_no_reg  val_r=0.355 val_rmse=1.519
=== Held-out year 2016 ===
  train hybrids: 4854, val hybrids: 556
  gblup        val_r=0.423 val_rmse=2.013


/content/drive/MyDrive/g2f_effect_decomposition/scripts/evaluation.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(pearsonr(y_true, y_pred)[0])


  mlp1_lasso   best val_r=0.446 (lambda=1e-02) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_sparse  best val_r=0.533 (lambda=1e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_l2      val_r=0.316 val_rmse=1.664
  mlp2_no_reg  val_r=0.424 val_rmse=1.707
=== Held-out year 2018 ===
  train hybrids: 4925, val hybrids: 1039
  gblup        val_r=0.241 val_rmse=1.776


...

  mlp1_lasso   best val_r=0.314 (lambda=1e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_sparse  best val_r=0.315 (lambda=1e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_l2      val_r=0.271 val_rmse=1.596
  mlp2_no_reg  val_r=0.149 val_rmse=1.589
=== Held-out year 2020 ===
  train hybrids: 4936, val hybrids: 1175
  gblup        val_r=0.769 val_rmse=1.464


/content/drive/MyDrive/g2f_effect_decomposition/scripts/evaluation.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(pearsonr(y_true, y_pred)[0])


  mlp1_lasso   best val_r=0.787 (lambda=5e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_sparse  best val_r=0.789 (lambda=5e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_l2      val_r=0.751 val_rmse=1.167
  mlp2_no_reg  val_r=0.761 val_rmse=1.244
=== Held-out year 2022 ===
  train hybrids: 4936, val hybrids: 549
  gblup        val_r=0.759 val_rmse=1.068


/content/drive/MyDrive/g2f_effect_decomposition/scripts/evaluation.py:19: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return float(pearsonr(y_true, y_pred)[0])


  mlp1_lasso   best val_r=0.739 (lambda=5e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_sparse  best val_r=0.754 (lambda=5e-03) -- full grid of 6 lambdas recorded, see comparison table below
  mlp2_l2      val_r=0.734 val_rmse=0.930
  mlp2_no_reg  val_r=0.749 val_rmse=0.962


## CV comparison table
Median train/val Pearson r and RMSE across the leave-one-year-out folds,
plus effective-marker count (GBLUP always uses all markers; mlp1_lasso/
mlp2_sparse's counts reflect the group-lasso sparsity penalty; mlp2_l2/
mlp2_no_reg won't show real sparsity -- L2 shrinks without zeroing, and
no-reg has no shrinkage at all -- their effective-marker counts are included
for completeness, not because either is expected to be sparse), same
accuracy-vs-interpretability framing as `nb06_mlp_variant_comparison.ipynb`.

In [10]:
comparison_rows = []

gblup_results = [r for r in fold_results if r['variant'] == 'gblup']
comparison_rows.append({
    'variant': 'gblup', 'l1_lambda': None,
    'median_train_r': np.median([r['train_metrics']['pearson_r'] for r in gblup_results]),
    'median_val_r': np.median([r['val_metrics']['pearson_r'] for r in gblup_results]),
    'median_val_rmse': np.median([r['val_metrics']['rmse'] for r in gblup_results]),
    'median_effective_markers': np.median([r['n_effective_markers'] for r in gblup_results]),
})

for variant in ['mlp1_lasso', 'mlp2_sparse']:
    for lam in L1_GRID:
        subset = [r for r in fold_results if r['variant'] == variant and r.get('l1_lambda') == lam]
        comparison_rows.append({
            'variant': variant, 'l1_lambda': lam,
            'median_train_r': np.median([r['train_metrics']['pearson_r'] for r in subset]),
            'median_val_r': np.median([r['val_metrics']['pearson_r'] for r in subset]),
            'median_val_rmse': np.median([r['val_metrics']['rmse'] for r in subset]),
            'median_effective_markers': np.median([r['n_effective_markers'] for r in subset]),
        })

for variant in ['mlp2_l2', 'mlp2_no_reg']:
    subset = [r for r in fold_results if r['variant'] == variant]
    comparison_rows.append({
        'variant': variant,
        'l1_lambda': MLP2_L2_LAMBDA if variant == 'mlp2_l2' else None,
        'median_train_r': np.median([r['train_metrics']['pearson_r'] for r in subset]),
        'median_val_r': np.median([r['val_metrics']['pearson_r'] for r in subset]),
        'median_val_rmse': np.median([r['val_metrics']['rmse'] for r in subset]),
        'median_effective_markers': np.median([r['n_effective_markers'] for r in subset]),
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table

,variant,l1_lambda,median_train_r,median_val_r,median_val_rmse,median_effective_markers
0,gblup,NaN,0.856441,0.423319,1.463785,2425.0
1,mlp1_lasso,0.00100,0.871323,0.385048,1.437452,2425.0
2,mlp1_lasso,0.00500,0.857394,0.439913,1.157327,2425.0
3,mlp1_lasso,0.01000,0.740890,0.446101,1.199943,2425.0
4,mlp1_lasso,0.02000,0.373177,0.208761,1.603229,2425.0
5,mlp1_lasso,0.03000,NaN,NaN,1.302152,0.0
6,mlp1_lasso,0.05000,NaN,NaN,1.301055,0.0
7,mlp2_sparse,0.00100,0.916132,0.533378,1.458436,2425.0
8,mlp2_sparse,0.00500,0.888764,0.503932,1.071828,2425.0
9,mlp2_sparse,0.01000,0.779704,0.421203,1.154260,2425.0


### Selecting lambda: the one-standard-error rule
Picking each variant's lambda by best validation Pearson r alone always
favors the weakest regularization in the grid, since some accuracy cost is
inherent to sparsity -- an accuracy-only rule would silently pick "none"
every time and defeat the point of asking for a sparse model. The
one-standard-error rule (standard lasso/glmnet practice) instead picks the
**sparsest** lambda whose median validation r is within one standard error
of the best lambda's -- i.e. "as sparse as possible without a
statistically meaningful accuracy cost." Applied to both MLP variants;
GBLUP's ridge alpha is still selected by best validation r alone, since
ridge doesn't zero out weights the same way L1 does -- there's no sparsity
axis to trade off there.

In [11]:
def select_lambda_one_se_rule(fold_results: list[dict], variant: str, l1_grid: list[float]) -> float:
    """Picks the sparsest (largest) lambda whose median validation Pearson r
    is within one standard error of the best lambda's median.
    """
    per_lambda = {}
    for lam in l1_grid:
        rs = [r['val_metrics']['pearson_r'] for r in fold_results
              if r['variant'] == variant and r.get('l1_lambda') == lam]
        rs = [x for x in rs if not np.isnan(x)]
        sem = np.std(rs) / np.sqrt(len(rs)) if len(rs) > 1 else 0.0
        per_lambda[lam] = (np.median(rs), sem)

    best_lambda = max(per_lambda, key=lambda l: per_lambda[l][0])
    best_median, best_sem = per_lambda[best_lambda]
    threshold = best_median - best_sem

    eligible = [lam for lam, (median, _) in per_lambda.items() if median >= threshold]
    chosen = max(eligible)
    print(f"{variant}: best lambda by accuracy alone = {best_lambda:.0e} (r={best_median:.3f}); "
          f"one-SE-rule chosen lambda = {chosen:.0e} "
          f"(r={per_lambda[chosen][0]:.3f}, threshold={threshold:.3f})")
    return chosen


selected_lambdas = {
    variant: select_lambda_one_se_rule(fold_results, variant, L1_GRID)
    for variant in ['mlp1_lasso', 'mlp2_sparse']
}

mlp1_lasso: best lambda by accuracy alone = 1e-02 (r=0.446); one-SE-rule chosen lambda = 1e-02 (r=0.446, threshold=0.321)
mlp2_sparse: best lambda by accuracy alone = 1e-03 (r=0.533); one-SE-rule chosen lambda = 5e-03 (r=0.504, threshold=0.445)


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


## Final fit on all training years + evaluation on the true 2024 test set
Refits each variant on all of 2014-2023 (using the hyperparameter that
performed best on average across the CV folds above), then evaluates
against `7_Testing_Observed_Values.csv` -- genuine held-out ground truth,
never touched during CV.

In [12]:
def most_common_gblup_alpha() -> float:
    values = [r['best_hparam'] for r in fold_results if r['variant'] == 'gblup']
    return float(pd.Series(values).median())


final_targets = build_hybrid_targets(trait_genotyped)
final_train_hybrids = final_targets.index.tolist()

test_observed_df['Hybrid'] = test_observed_df['Hybrid'].astype(str)
test_targets = test_observed_df.groupby('Hybrid')['Yield_Mg_ha'].mean()
test_hybrids = [h for h in test_targets.index if h in geno_num.index]

marker_median_final = geno_num.loc[final_train_hybrids].median()
geno_imputed_final = geno_num.fillna(marker_median_final)

X_train_final_raw = geno_imputed_final.loc[final_train_hybrids].to_numpy(dtype=np.float64)
X_test_final_raw = geno_imputed_final.loc[test_hybrids].to_numpy(dtype=np.float64)

marker_mean_final = X_train_final_raw.mean(axis=0)
marker_std_final = X_train_final_raw.std(axis=0)
marker_std_final[marker_std_final == 0] = 1.0
X_train_final_std = (X_train_final_raw - marker_mean_final) / marker_std_final
X_test_final_std = (X_test_final_raw - marker_mean_final) / marker_std_final

y_train_final = final_targets.loc[final_train_hybrids, 'hybrid_mean_yield'].to_numpy(dtype=np.float32)
y_test_final = test_targets.loc[test_hybrids].to_numpy(dtype=np.float32)
weights_train_final = reliability_weights(
    final_targets.loc[final_train_hybrids, 'n_envs_tested'].to_numpy())

print(f"Final train hybrids: {len(final_train_hybrids)} | Test hybrids (genotyped): {len(test_hybrids)}")

Final train hybrids: 4938 | Test hybrids (genotyped): 1063


In [13]:
final_results = {}

# GBLUP
allele_freq_final = vanraden_allele_freq(X_train_final_raw)
denom_final = vanraden_denominator(allele_freq_final)
G_train_final = vanraden_kernel(X_train_final_raw, X_train_final_raw, allele_freq_final, denom_final)
G_test_final = vanraden_kernel(X_test_final_raw, X_train_final_raw, allele_freq_final, denom_final)

gblup_alpha = most_common_gblup_alpha()
from scripts.training import GBLUPModel
gblup_model = GBLUPModel(alpha=gblup_alpha)
gblup_model.fit(G_train_final, y_train_final, sample_weight=weights_train_final)

final_results['gblup'] = {
    'hparam': gblup_alpha,
    'train_metrics': evaluate_predictions(y_train_final, gblup_model.predict(G_train_final)),
    'test_metrics': evaluate_predictions(y_test_final, gblup_model.predict(G_test_final)),
    'n_effective_markers': X_train_final_raw.shape[1],
}

weights_val_dummy = np.ones(len(y_test_final), dtype=np.float32)

# Group-lasso MLP variants (SGD + proximal)
for variant_name in ['mlp1_lasso', 'mlp2_sparse']:
    lam = selected_lambdas[variant_name]
    model, l1_idx = build_variant(variant_name, X_train_final_std.shape[1], HIDDEN_DIMS_MLP1, HIDDEN_DIMS_MLP2)
    train_loader = make_loader(X_train_final_std, y_train_final, weights_train_final, BATCH_SIZE, shuffle=True)
    test_loader = make_loader(X_test_final_std, y_test_final, weights_val_dummy, BATCH_SIZE, shuffle=False)
    model, _ = fit_mlp(model, train_loader, test_loader, l1_idx, lam, LR, NUM_EPOCHS, PATIENCE, DEVICE,
                        checkpoint_path=MODELS_DIR / f'{variant_name}_final.pt')

    model.eval()
    with torch.no_grad():
        pred_train = model(torch.as_tensor(X_train_final_std, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        pred_test = model(torch.as_tensor(X_test_final_std, dtype=torch.float32).to(DEVICE)).cpu().numpy()

    final_results[variant_name] = {
        'hparam': lam,
        'train_metrics': evaluate_predictions(y_train_final, pred_train),
        'test_metrics': evaluate_predictions(y_test_final, pred_test),
        'n_effective_markers': effective_markers(model, layer_idx=0),
    }

# Smooth-penalty MLP variants (Adam)
for variant_name in ['mlp2_l2', 'mlp2_no_reg']:
    model, layer_idx = build_variant(variant_name, X_train_final_std.shape[1], HIDDEN_DIMS_MLP1, HIDDEN_DIMS_MLP2)
    train_loader = make_loader(X_train_final_std, y_train_final, weights_train_final, BATCH_SIZE, shuffle=True)
    test_loader = make_loader(X_test_final_std, y_test_final, weights_val_dummy, BATCH_SIZE, shuffle=False)
    penalty_fn = (lambda m: l2_penalty(m, layer_idx, MLP2_L2_LAMBDA)) if variant_name == 'mlp2_l2' else None
    model, _ = fit_mlp_adam(model, train_loader, test_loader, penalty_fn, ADAM_LR, NUM_EPOCHS, PATIENCE, DEVICE,
                             checkpoint_path=MODELS_DIR / f'{variant_name}_final.pt')

    model.eval()
    with torch.no_grad():
        pred_train = model(torch.as_tensor(X_train_final_std, dtype=torch.float32).to(DEVICE)).cpu().numpy()
        pred_test = model(torch.as_tensor(X_test_final_std, dtype=torch.float32).to(DEVICE)).cpu().numpy()

    final_results[variant_name] = {
        'hparam': MLP2_L2_LAMBDA if variant_name == 'mlp2_l2' else None,
        'train_metrics': evaluate_predictions(y_train_final, pred_train),
        'test_metrics': evaluate_predictions(y_test_final, pred_test),
        'n_effective_markers': effective_markers(model, layer_idx=0),
    }

final_comparison = pd.DataFrame([
    {
        'variant': name,
        'hparam': res['hparam'],
        'train_pearson_r': res['train_metrics']['pearson_r'],
        'test_pearson_r': res['test_metrics']['pearson_r'],
        'train_rmse': res['train_metrics']['rmse'],
        'test_rmse': res['test_metrics']['rmse'],
        'n_effective_markers': res['n_effective_markers'],
    }
    for name, res in final_results.items()
])
final_comparison

  Epoch 0: train_loss=66.4091, val_loss=52.2868
  Epoch 10: train_loss=1.3054, val_loss=14.6851
  Epoch 20: train_loss=6.4319, val_loss=36.1796
  Early stopping at epoch 25
  Epoch 0: train_loss=76.1289, val_loss=58.5747
  Epoch 10: train_loss=0.3156, val_loss=5.3764
  Epoch 20: train_loss=0.3167, val_loss=4.8121
  Epoch 30: train_loss=0.3713, val_loss=5.7570
  Early stopping at epoch 38
  Epoch 0: train_loss=33.9024, val_loss=10.3347
  Epoch 10: train_loss=0.3805, val_loss=6.5454
  Epoch 20: train_loss=0.1962, val_loss=5.2772
  Epoch 30: train_loss=0.1242, val_loss=5.1377
  Epoch 40: train_loss=0.0800, val_loss=5.1202
  Epoch 50: train_loss=0.0646, val_loss=4.5927
  Epoch 60: train_loss=0.0581, val_loss=5.3418
  Epoch 70: train_loss=0.0656, val_loss=5.4195
  Epoch 80: train_loss=0.0305, val_loss=5.1908
  Early stopping at epoch 85
  Epoch 0: train_loss=26.3704, val_loss=7.1885
  Epoch 10: train_loss=0.3549, val_loss=6.4187
  Epoch 20: train_loss=0.1824, val_loss=5.6111
  Epoch 30: tra

,variant,hparam,train_pearson_r,test_pearson_r,train_rmse,test_rmse,n_effective_markers
0,gblup,0.10000,0.848399,0.229108,0.748366,1.441267,2425
1,mlp1_lasso,0.01000,0.733424,0.145376,1.276301,2.886420,2425
2,mlp2_sparse,0.00500,0.881365,0.199295,0.890836,2.118757,2425
3,mlp2_l2,0.00001,0.973641,0.185059,0.312560,1.989960,2424
4,mlp2_no_reg,NaN,0.958926,0.155396,0.385916,1.925033,2425


## Save results

In [14]:
fold_results_df = pd.DataFrame([
    {
        'variant': r['variant'], 'held_out_year': r['held_out_year'],
        'hparam': r.get('best_hparam', r.get('l1_lambda')),  # ridge alpha for gblup, l1_lambda for MLPs
        'train_pearson_r': r['train_metrics']['pearson_r'], 'val_pearson_r': r['val_metrics']['pearson_r'],
        'train_rmse': r['train_metrics']['rmse'], 'val_rmse': r['val_metrics']['rmse'],
        'n_effective_markers': r['n_effective_markers'],
    }
    for r in fold_results
])

fold_results_df.to_csv(RESULTS_DIR / 'cv_fold_results.csv', index=False)
comparison_table.to_csv(RESULTS_DIR / 'cv_comparison_table.csv', index=False)
final_comparison.to_csv(RESULTS_DIR / 'final_test_comparison.csv', index=False)

print(f"Saved to {RESULTS_DIR}")
print("  cv_fold_results.csv       -- per-fold, per-variant metrics")
print("  cv_comparison_table.csv   -- median CV metrics per variant")
print("  final_test_comparison.csv -- final train/test metrics on the real 2024 holdout")

Saved to /content/drive/MyDrive/g2f_effect_decomposition/results/genotype_model
  cv_fold_results.csv       -- per-fold, per-variant metrics
  cv_comparison_table.csv   -- median CV metrics per variant
  final_test_comparison.csv -- final train/test metrics on the real 2024 holdout


## Summary
Fill in after running:
- Best CV val_pearson_r variant:
- Best final test_pearson_r variant:
- Does GBLUP's baseline hold up against the MLPs, or do they meaningfully beat it?
- MLP-1 vs MLP-2: does the second hidden layer earn its added complexity?
- **Regularization-type ablation** (mlp2_sparse vs mlp2_l2 vs mlp2_no_reg,
  same architecture): does group-lasso sparsity cost accuracy relative to
  L2 or no regularization, or does it hold up despite the sparsity
  constraint? Does mlp2_no_reg overfit visibly worse (larger train/val gap)
  than the two regularized variants?
- Effective-marker counts: how sparse did the group-lasso penalty actually
  make mlp1_lasso/mlp2_sparse? (mlp2_l2/mlp2_no_reg aren't expected to show
  real sparsity -- that's not what those variants are testing.)
- Any variant with a large train/val or train/test gap (overfitting despite
  weighting/regularization)?
- This model's "vote" (final_results predictions) is the genotype-alone input
  to Phase 3's diagnostic/orthogonalization layer -- carry forward whichever
  variant wins on val_pearson_r, not train_pearson_r.